# Heavy Traffic Indicators on I-94

A reproducible look at hourly westbound traffic near Minneapolis and St. Paul. Run `python scripts/download_data.py` from the repository root before executing this notebook. All charts and tables are also published in `results/`.

## Load and validate

The source repeats some timestamps for multiple weather descriptions. All duplicate timestamps must have the same traffic count. We keep one traffic observation per hour so repeated descriptions do not increase that hour's weight.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.traffic_analysis import load_traffic, grouped_volume

hourly, quality = load_traffic(ROOT / 'data' / 'Metro_Interstate_Traffic_Volume.csv')
quality

## Weekdays and weekends

The weekday mean is roughly 36% higher. This is an observed association across the dataset, not an estimate of a day-type effect after controlling for other factors.

In [ ]:
day_type = grouped_volume(hourly, ['day_type'])
day_type

## Hourly pattern

Weekdays have clear morning and late-afternoon peaks. Weekend traffic rises later and has a broad midday plateau. Each point averages only the observed hours at that clock time.

In [ ]:
hourly_pattern = grouped_volume(hourly, ['day_type', 'hour'])
fig, ax = plt.subplots(figsize=(10, 5))
for label, subset in hourly_pattern.groupby('day_type'):
    ax.plot(subset['hour'], subset['mean_vehicles'], marker='o', label=label)
ax.set(xlabel='Hour of day (local time)', ylabel='Mean vehicles per observed hour', title='Traffic by hour and day type')
ax.legend()
plt.show()

![Published hourly chart](../results/hourly_patterns.png)

## Monthly comparison

The source covers months unevenly across years. The following are descriptive averages, not adjusted seasonal estimates.

In [ ]:
monthly = grouped_volume(hourly, ['month'])
monthly

![Published monthly chart](../results/monthly_patterns.png)

## Interpretation limits

This station measures one westbound direction at one location. Missing hours, changing conditions over six years, holidays, and daylight saving time can affect comparisons. Multiple weather labels can refer to the same hour, so a naive weather grouping is misleading. No causal weather claim is made here. See the README for the source citation and reproduction steps.